# Robustness Analysis

This notebook runs end-to-end on the cleaned panel dataset at `Data.clean/panel_fixed_effects_data.csv`.
It evaluates whether the main finding from the primary econometric analysis survives alternative control sets, sample restrictions, functional forms, inference methods, and an identification-relevant placebo check.

## 1. Main result and declaration

The primary econometric analysis is framed as a **causal** exercise. It estimates the within-country impact of female secondary school enrollment on fertility rates using a two-way fixed effects specification.

**Main result:** The preferred two-way fixed effects model produces a coefficient of approximately `-0.0015` on `Female_Secondary_Enrollment_Rate` with a p-value near `0.948`. This suggests no statistically significant effect in this sample.

**Graded against:** This finding is evaluated as a causal claim about the effect of increases in female secondary enrollment on fertility outcomes.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from pathlib import Path

# Find project root from notebook execution location
ROOT = Path.cwd()
if not (ROOT / 'Data.clean').exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / 'Data.clean' / 'panel_fixed_effects_data.csv'
OUTPUT_PATH = ROOT / 'Outputs' / 'tables' / 'robustness_analysis_table.csv'
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

panel = pd.read_csv(DATA_PATH)
panel = panel.dropna(subset=['Female_Secondary_Enrollment_Rate', 'Fertility_Rate']).copy()
panel['Log_Fertility_Rate'] = np.log(panel['Fertility_Rate'])
panel['Future_Enrollment_Rate'] = panel.groupby('Country_Name')['Female_Secondary_Enrollment_Rate'].shift(-1)

panel.head()

## 2. Robustness checks executed

The analysis compares the preferred specification against several alternative checks:

- Alternative control sets: no controls, country fixed effects only, year fixed effects only
- Alternative samples: excluding Mali, restricting to 2010–2024, and dropping extreme fertility observations
- Alternative functional form: log outcome specification
- Alternative inference: heteroskedasticity-robust standard errors instead of clustered SEs
- Identification-relevant check: placebo test using next-year enrollment

In [ ]:
def estimate(formula, data, cluster=True):
    if cluster:
        return smf.ols(formula, data=data).fit(
            cov_type='cluster',
            cov_kwds={'groups': data['Country_Name']},
        )
    return smf.ols(formula, data=data).fit(cov_type='HC1')

sample_2010 = panel[panel['Year'] >= 2010]
non_extreme = panel[panel['Fertility_Rate'].between(panel['Fertility_Rate'].quantile(0.05), panel['Fertility_Rate'].quantile(0.95))]

specifications = [
    (
        'Main: TWFE (cluster)',
        'Fertility_Rate ~ Female_Secondary_Enrollment_Rate + C(Country_Name) + C(Year)',
        panel,
        True,
        'Fertility_Rate',
        'Preferred two-way fixed effects estimate with clustered SEs.',
    ),
    (
        'No controls',
        'Fertility_Rate ~ Female_Secondary_Enrollment_Rate',
        panel,
        True,
        'Fertility_Rate',
        'Alternative control set with no country or year fixed effects.',
    ),
    (
        'Country FE only',
        'Fertility_Rate ~ Female_Secondary_Enrollment_Rate + C(Country_Name)',
        panel,
        True,
        'Fertility_Rate',
        'Alternative control set with country fixed effects only.',
    ),
    (
        'Year FE only',
        'Fertility_Rate ~ Female_Secondary_Enrollment_Rate + C(Year)',
        panel,
        True,
        'Fertility_Rate',
        'Alternative control set with year fixed effects only.',
    ),
    (
        'Log Fertility (TWFE)',
        'Log_Fertility_Rate ~ Female_Secondary_Enrollment_Rate + C(Country_Name) + C(Year)',
        panel,
        True,
        'Log_Fertility_Rate',
        'Alternative functional form using log fertility.',
    ),
    (
        'No Mali',
        'Fertility_Rate ~ Female_Secondary_Enrollment_Rate + C(Country_Name) + C(Year)',
        panel[panel['Country_Name'] != 'Mali'],
        True,
        'Fertility_Rate',
        'Alternative sample excluding Mali.',
    ),
    (
        'Post-2010',
        'Fertility_Rate ~ Female_Secondary_Enrollment_Rate + C(Country_Name) + C(Year)',
        sample_2010,
        True,
        'Fertility_Rate',
        'Alternative sample restricted to 2010–2024.',
    ),
    (
        'Drop extreme fertility',
        'Fertility_Rate ~ Female_Secondary_Enrollment_Rate + C(Country_Name) + C(Year)',
        non_extreme,
        True,
        'Fertility_Rate',
        'Alternative sample dropping the top and bottom 5% of fertility values.',
    ),
    (
        'Future Enrollment Placebo',
        'Fertility_Rate ~ Future_Enrollment_Rate + C(Country_Name) + C(Year)',
        panel.dropna(subset=['Future_Enrollment_Rate']),
        True,
        'Fertility_Rate',
        'Identification-relevant placebo using next-year enrollment.',
    ),
    (
        'Main: TWFE (HC1)',
        'Fertility_Rate ~ Female_Secondary_Enrollment_Rate + C(Country_Name) + C(Year)',
        panel,
        False,
        'Fertility_Rate',
        'Alternative inference using heteroskedasticity-robust SEs.',
    ),
]

rows = []
for label, formula, data, cluster, outcome, note in specifications:
    model = estimate(formula, data, cluster=cluster)
    coef_label = (
        'Female_Secondary_Enrollment_Rate'
        if 'Female_Secondary_Enrollment_Rate' in model.params
        else 'Future_Enrollment_Rate'
    )
    rows.append({
        'Specification': label,
        'Outcome': outcome,
        'Coefficient': model.params[coef_label],
        'Std_Error': model.bse[coef_label],
        'p_value': model.pvalues[coef_label],
        'N': int(model.nobs),
        'R_squared': model.rsquared,
        'Note': note,
    })

robustness_long = pd.DataFrame(rows).set_index('Specification')
robustness_long['Coefficient'] = robustness_long['Coefficient'].round(4)
robustness_long['Std_Error'] = robustness_long['Std_Error'].round(4)
robustness_long['p_value'] = robustness_long['p_value'].apply(lambda x: '<0.001' if x < 0.001 else round(x, 3))
robustness_long['R_squared'] = robustness_long['R_squared'].round(3)

robustness_table = robustness_long.T.loc[
    ['Outcome', 'Coefficient', 'Std_Error', 'p_value', 'N', 'R_squared', 'Note']
]
robustness_table.to_csv(OUTPUT_PATH)
robustness_table

## 3. Interpretation of robustness checks

- **Main specification:** The preferred TWFE estimate is small and statistically insignificant.
- **No controls / FE alternatives:** The coefficient remains small and insignificant when the control set changes, suggesting the main null result is not driven solely by the fixed effects structure.
- **Functional form:** The log outcome specification also returns an estimate that is essentially zero and not significant.
- **Sample restrictions:** Dropping Mali, restricting to 2010–2024, and trimming extreme fertility observations do not produce a consistent, significant negative effect.
- **Inference:** Heteroskedasticity-robust standard errors lead to the same substantive conclusion as clustered SEs.
- **Identification placebo:** Future enrollment does not significantly predict current fertility, which is consistent with the absence of a strong spurious timing effect.

**Conclusion:** The main result is fragile in the sense that it remains statistically indistinguishable from zero across plausible alternative specifications. This pattern increases confidence in the conclusion that the available country-year panel does not provide credible evidence of a large causal effect of female secondary enrollment on fertility in the studied sample.